# DM1 Project 2026/27 — Modulo 1: Data Understanding & Preparation

**Dataset**: *Seller-Level Vehicle Dataset* (`seller_info_dm1_2627.csv`) · **Target**: `seller_rating_class`

Notebook della parte 1 (30 punti), organizzato secondo la scaletta `docs/Scaletta_Parte1_DU_DP.md`.

| Sezione | Blocco delle guideline (slide 7) |
|---|---|
| 1. Setup e caricamento | — |
| 2. Data semantics | *Data Semantics* |
| 3. Riparazione dei numeri corrotti | *Assessing data quality* |
| 4. Distribuzioni e statistiche | *Distribution of the variables and statistics* |
| 5. Qualità dei dati | *Assessing data quality* |
| 6. Valori mancanti | *Assessing data quality* |
| 7. Outlier, trasformazioni, nuove variabili | *Possible variable transformation* |
| 8. Correlazioni ed eliminazione di variabili | *Pairwise correlations and eventual elimination* |
| 9. Dataset finali e riepilogo delle decisioni | ponte verso i moduli successivi |

**Come leggere il notebook**
- Prima di ogni blocco di codice, una cella *"Cosa fa il codice qui sotto"* spiega i passaggi e le funzioni usate.
- Le celle *"Commento per il report"* contengono l'interpretazione dei risultati, da riportare (con parole vostre) nel report.
- Il notebook va eseguito dalla cartella `notebooks/` (è quello che fa VS Code di default): i percorsi dei file iniziano con `../`, che vuol dire "la cartella sopra".
- Nel codice, `df` è la tabella dei dati (un *DataFrame* di pandas) e `df["nome"]` è una sua colonna.

> **Nota sull'ordine.** La riparazione dei numeri corrotti (§3) viene **prima** delle statistiche (§4): senza di essa 7 colonne numeriche sarebbero testo e resterebbero fuori dalle analisi. Nel report l'ordine può seguire le guideline.

## 1. Setup e caricamento

> **Cosa fa il codice qui sotto.**
> - Importa le librerie che servono: `pandas` (tabelle), `numpy` (calcoli), `matplotlib` e `seaborn` (grafici), `scipy` (test χ²), `scikit-learn` (PCA, regressione lineare, standardizzazione). `as pd`, `as np`... sono solo abbreviazioni per scrivere meno.
> - `sns.set_theme()` attiva lo stile grafico di base di seaborn.
> - Crea la variabile `classi`, la lista delle classi del target nel loro ordine naturale: la useremo nei grafici per mostrarle sempre nell'ordine low → medium → high.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

sns.set_theme()
pd.set_option("display.max_columns", 50)     # mostra fino a 50 colonne quando stampiamo una tabella

classi = ["low", "medium", "high"]           # classi del target, in ordine

> **Cosa fa il codice qui sotto.**
> - `pd.read_csv` legge il file e lo mette nella tabella `df_raw` ("raw" = dati grezzi, che non modificheremo). `sep=";"` dice che le colonne sono separate da punto e virgola: senza, pandas leggerebbe tutto come un'unica colonna.
> - `.shape` dà (numero di righe, numero di colonne).
> - `.duplicated().sum()` conta le righe identiche a una riga precedente; applicato alla sola colonna `sp_id` conta gli ID ripetuti.
> - `.head(3)` mostra le prime 3 righe.

In [ ]:
df_raw = pd.read_csv("../data/seller_info_dm1_2627.csv", sep=";")

print("Dimensioni:", df_raw.shape)
print("Righe duplicate:", df_raw.duplicated().sum())
print("sp_id duplicati:", df_raw["sp_id"].duplicated().sum())
df_raw.head(3)

> **Cosa fa il codice qui sotto.** Costruisce una piccola tabella con una riga per colonna del dataset:
> - `dtypes`: il tipo che pandas ha assegnato (`int64` = intero, `float64` = decimale, `str` = testo);
> - `isna().sum()`: quanti valori mancanti (NaN) ci sono;
> - `nunique()`: quanti valori diversi ci sono.
>
> Serve a scoprire le colonne lette con il tipo sbagliato (es. numeri letti come testo).

In [ ]:
pd.DataFrame({
    "tipo pandas": df_raw.dtypes,
    "n_NaN": df_raw.isna().sum(),
    "n_distinti": df_raw.nunique(),
})

**Commento per il report.** 13.843 venditori × 33 attributi, nessun duplicato (né di riga né di `sp_id`). Ogni riga è **un venditore**: i valori sono statistiche **aggregate** sui suoi annunci (mediane, proporzioni, mode). `sp_id` è letto come numero decimale ma è un identificativo; 7 attributi numerici sono letti come testo a causa di valori corrotti (§3).

## 2. Data semantics

Ogni attributo si classifica secondo le scale di Stevens (DM1.01 §3): **nominale** (solo =, ≠), **ordinale** (anche <, >), **intervallo** (anche differenze, zero arbitrario), **rapporto** (anche rapporti, zero assoluto).

Concetti chiave per questo dataset:
- le `*_median` sono **statistiche di secondo livello**: descrivono l'auto *tipica* dell'inventario, non una singola auto;
- le `pct_*` sono **proporzioni** in [0, 1];
- le `dominant_*` sono **mode** dell'inventario: un venditore con 51% SUV e 49% sedan risulta "suv";
- `year_median` è su scala a **intervallo** (l'anno 0 è convenzionale), mentre `vehicle_age_median` è su scala a **rapporto**;
- il target è **ordinale** (low < medium < high).

> **Cosa fa il codice qui sotto.**
> - Scrive a mano la tabella semantica: ogni riga tra parentesi tonde è un attributo con (nome, significato, scala di Stevens, ruolo). `pd.DataFrame(...)` trasforma l'elenco in una tabella, con i nomi di colonna indicati in `columns`.
> - `value_counts()` conta quante volte compare ogni valore di una colonna: qui, quanti attributi ci sono per ogni scala.

In [ ]:
semantica = pd.DataFrame([
    ("sp_id", "ID del venditore", "nominale", "ID"),
    ("n_listings", "n. totale di annunci", "rapporto", "predittore"),
    ("n_listing_days", "giorni distinti con annunci", "rapporto", "predittore"),
    ("listing_span_days", "giorni tra primo e ultimo annuncio", "rapporto", "predittore"),
    ("price_median", "prezzo mediano (USD)", "rapporto", "predittore"),
    ("savings_amount_median", "risparmio mediano pubblicizzato (USD)", "rapporto", "predittore"),
    ("daysonmarket_median", "giorni mediani sul mercato", "rapporto", "predittore"),
    ("mileage_median", "chilometraggio mediano (miglia)", "rapporto", "predittore"),
    ("year_median", "anno-modello mediano", "intervallo", "predittore"),
    ("vehicle_age_median", "età mediana all'annuncio (anni)", "rapporto", "predittore"),
    ("owner_count_median", "n. mediano di proprietari precedenti", "rapporto", "predittore"),
    ("horsepower_median", "potenza mediana (hp)", "rapporto", "predittore"),
    ("engine_displacement_median", "cilindrata mediana (cc)", "rapporto", "predittore"),
    ("city_fuel_economy_median", "consumo urbano (MPG)", "rapporto", "predittore"),
    ("highway_fuel_economy_median", "consumo extraurbano (MPG)", "rapporto", "predittore"),
    ("back_legroom_median", "spazio gambe posteriore (pollici)", "rapporto", "predittore"),
    ("front_legroom_median", "spazio gambe anteriore (pollici)", "rapporto", "predittore"),
    ("fuel_tank_volume_median", "serbatoio (galloni)", "rapporto", "predittore"),
    ("height_median", "altezza (pollici)", "rapporto", "predittore"),
    ("length_median", "lunghezza (pollici)", "rapporto", "predittore"),
    ("width_median", "larghezza (pollici)", "rapporto", "predittore"),
    ("pct_isCab", "quota di ex taxi", "rapporto", "predittore"),
    ("pct_is_new", "quota di auto nuove", "rapporto", "predittore"),
    ("pct_salvage", "quota di auto incidentate", "rapporto", "predittore"),
    ("pct_theft_title", "quota con titolo di furto", "rapporto", "predittore"),
    ("dominant_body_type", "carrozzeria più frequente", "nominale", "predittore"),
    ("dominant_fuel_type", "alimentazione più frequente", "nominale", "predittore"),
    ("dominant_transmission", "cambio più frequente", "nominale", "predittore"),
    ("dominant_wheel_system", "trazione più frequente", "nominale", "predittore"),
    ("dominant_listing_color", "colore più frequente", "nominale", "predittore"),
    ("franchise_status", "concessionario ufficiale o indipendente", "nominale", "predittore"),
    ("new_used_profile", "composizione nuovo/usato", "nominale", "predittore (derivata)"),
    ("seller_rating_class", "classe di rating del venditore", "ordinale", "target"),
], columns=["attributo", "significato", "scala", "ruolo"])

print(semantica["scala"].value_counts())
semantica

> **Cosa fa il codice qui sotto.** Con `value_counts()` mostra quali valori compaiono davvero in `franchise_status` e `new_used_profile`, e quante volte. Va confrontato con la documentazione, che ne dichiara 4 ciascuna.

In [ ]:
print(df_raw["franchise_status"].value_counts(), "\n")
print(df_raw["new_used_profile"].value_counts())

> **Cosa fa il codice qui sotto.**
> - `pd.cut` divide `pct_is_new` in tre fasce: esattamente 0, tra 0 e 1, esattamente 1 (i numeri tra parentesi quadre sono i confini delle fasce).
> - `pd.crosstab` costruisce una tabella di contingenza: conta i venditori per ogni combinazione (profilo, fascia). Se ogni profilo cade in una sola fascia, `new_used_profile` è ricavata da `pct_is_new`.

In [ ]:
fascia = pd.cut(df_raw["pct_is_new"], [-0.01, 0, 0.999999, 1], labels=["0", "tra 0 e 1", "1"])
pd.crosstab(df_raw["new_used_profile"], fascia)

> **Cosa fa il codice qui sotto.** Somma, riga per riga, l'anno del modello e l'età del veicolo: il risultato è l'anno in cui è stato pubblicato l'annuncio. `describe()` ne riassume la distribuzione (media, deviazione standard, minimo, quartili, massimo).

In [ ]:
anno_annuncio = df_raw["year_median"] + df_raw["vehicle_age_median"]
anno_annuncio.describe()

**Commento per il report.**
- La documentazione elenca 4 valori per `franchise_status` e `new_used_profile`, ma nei dati **"mixed" e "unknown" di `franchise_status` non compaiono** e **"unknown" di `new_used_profile` non compare**: c'è una discrepanza tra documentazione e dati.
- `new_used_profile` è **perfettamente determinata** da `pct_is_new` (0 → used_only, 1 → new_only, altrimenti mixed): è un attributo derivato, quindi ridondante (§8).
- `year_median + vehicle_age_median ≈ 2020` (dev. std ≈ 0,4): gli annunci sono stati raccolti intorno al 2020 (*timeliness*) e le due variabili sono ridondanti per costruzione.

### 2.5 Approfondimento: Ridondanza e correlazioni del blocco Età, Chilometraggio e Profilo

> **Cosa fa il codice qui sotto.**
> - Isola il blocco di feature legate a età e utilizzo del parco veicoli: `mileage_median`, `year_median`, `vehicle_age_median`, `owner_count_median`, `pct_is_new` e `new_used_profile`.
> - Calcola la matrice di correlazione lineare di Pearson e quella per ranghi di Spearman per quantificare la collinearità.
> - Calcola la *Correlation Ratio* ($\eta$) per misurare l'associazione tra la categorica `new_used_profile` e le numeriche.
> - Disegna un grafico a 4 pannelli diagnostici (salvato in `report/figures/`):
>   1. **Heatmap Spearman**: correlazione monotona tra le continue.
>   2. **Scatter plot**: perfetta collinearità inversa ($r = -0.996$) tra `year_median` e `vehicle_age_median`.
>   3. **Boxplot**: corrispondenza esatta tra `pct_is_new` e le categorie di `new_used_profile`.
>   4. **Boxplot chilometraggio (log)**: separazione marcata tra venditori di solo usato, misti e solo nuovo.

In [ ]:
# Blocco di feature su età, chilometraggio e quota nuovo
cols_ridondanza = [
    "mileage_median",
    "year_median",
    "vehicle_age_median",
    "owner_count_median",
    "pct_is_new",
]
col_profilo = "new_used_profile"

# Dataset di lavoro (usa df se già pulito, altrimenti df_raw)
dati_blocco = (df if "df" in locals() else df_raw)[cols_ridondanza + [col_profilo]].dropna()

# 1. Matrici di correlazione: Pearson (lineare) e Spearman (monotona / per ranghi)
corr_pearson = dati_blocco[cols_ridondanza].corr(method="pearson").round(3)
corr_spearman = dati_blocco[cols_ridondanza].corr(method="spearman").round(3)

print("=== Correlazione Lineare di Pearson ===")
display(corr_pearson)

print("\n=== Correlazione per Ranghi di Spearman ===")
display(corr_spearman)

# 2. Correlation Ratio (Eta) tra categorica e numeriche: quantifica quanta varianza è spiegata
print("\n=== Correlation Ratio (Eta) con new_used_profile ===")
for col in cols_ridondanza:
    grand_mean = dati_blocco[col].mean()
    ss_total = ((dati_blocco[col] - grand_mean) ** 2).sum()
    group_means = dati_blocco.groupby(col_profilo)[col].mean()
    group_counts = dati_blocco.groupby(col_profilo)[col].count()
    ss_between = (group_counts * ((group_means - grand_mean) ** 2)).sum()
    eta = (ss_between / ss_total) ** 0.5 if ss_total > 0 else 0.0
    print(f"* Eta(new_used_profile, {col:<20}) = {eta:.3f}")

# 3. Grafici diagnostici a 4 pannelli
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# A: Heatmap Spearman
sns.heatmap(
    corr_spearman, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, square=True, ax=axes[0, 0]
)
axes[0, 0].set_title("A. Correlazione di Spearman (Monotonia)", fontsize=11, fontweight="bold")

# B: Scatter plot collinearità year vs vehicle_age
campione_scatter = dati_blocco.sample(n=min(2000, len(dati_blocco)), random_state=42)
sns.scatterplot(
    data=campione_scatter, x="year_median", y="vehicle_age_median",
    alpha=0.35, color="#2a78d6", ax=axes[0, 1]
)
axes[0, 1].set_title("B. Collinearità Inversa Perfetta (r = -0.996)\nyear_median vs vehicle_age_median", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel("Anno Mediano Modello (year_median)")
axes[0, 1].set_ylabel("Anzianità Mediana Veicolo (vehicle_age_median)")

# C: Boxplot pct_is_new vs new_used_profile
sns.boxplot(
    data=dati_blocco, x=col_profilo, y="pct_is_new",
    hue=col_profilo, palette=["#2a78d6", "#eb6834", "#1baf7a"], legend=False, ax=axes[1, 0]
)
axes[1, 0].set_title("C. Relazione esatta: pct_is_new vs new_used_profile", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel("new_used_profile")
axes[1, 0].set_ylabel("Quota Auto Nuove (pct_is_new)")

# D: Boxplot chilometraggio (log) per profilo
log_km = np.log1p(dati_blocco["mileage_median"])
sns.boxplot(
    x=dati_blocco[col_profilo], y=log_km,
    hue=dati_blocco[col_profilo], palette=["#2a78d6", "#eb6834", "#1baf7a"], legend=False, ax=axes[1, 1]
)
axes[1, 1].set_title("D. Chilometraggio log(1 + km) per Profilo Venditore", fontsize=11, fontweight="bold")
axes[1, 1].set_xlabel("new_used_profile")
axes[1, 1].set_ylabel("log(1 + mileage_median)")

plt.tight_layout()
plt.savefig("../report/figures/fig02_ridondanza_veicoli.png", dpi=150, bbox_inches="tight")
plt.show()

**Commento per il report (Decisioni di eliminazione della ridondanza).**
- **`year_median` vs `vehicle_age_median` (Eliminazione di `year_median`):** correlazione lineare $r = -0.996$ (Spearman $\rho = -0.984$). Poiché $\text{vehicle\_age\_median} \approx 2020 - \text{year\_median}$, una variabile è la traslazione speculare dell'altra. Si decide di **eliminare `year_median`** e mantenere `vehicle_age_median` in quanto l'età è su scala di rapporto con zero naturale, ideale per distanze euclidee e clustering.
- **`pct_is_new` vs `new_used_profile` (Esclusione di `new_used_profile` dai modelli):** il profilo categorico è una discretizzazione deterministica rigida (`pct == 0` $\rightarrow$ *used_only*, `pct == 1` $\rightarrow$ *new_only*, `0 < pct < 1` $\rightarrow$ *mixed*). Includere entrambe raddoppierebbe artificialmente il peso dell'informazione. Si usa **`pct_is_new`** nelle funzioni di distanza del clustering e si mantiene `new_used_profile` solo per descrivere e profilare i cluster a posteriori.
- **`mileage_median` e `owner_count_median`:** `mileage_median` ha una correlazione monotona fortissima con l'età (Spearman $\rho = 0.927$) e separa nettamente i venditori di solo usato (mediana ~93.000 miglia) da quelli di nuovo/misto (~4–18 miglia). Viene conservata ma necessita di standardizzazione $z$-score e trasformazione $\log(1+x)$ per mitigare la forte asimmetria.

### 2.6 Audit preliminare di qualità sintattica: rilevamento e conteggio delle "stringhe giganti"

> **Cosa fa il codice qui sotto.**
> - Prima di applicare qualsiasi funzione di pulizia o correzione, esegue un audit sulle 7 colonne numeriche caricate come testo (`object`) da pandas.
> - Rileva e conta quanti valori contengono stringhe anomale con separatori a punto multiplo (es. `38.650.000.000.000.000` o `26.037.495.000.000.000`), che rappresentano valori fisicamente impossibili ed enormi.
> - Estrae e mostra per ciascuna colonna 3 esempi reali di valori giganti non elaborati estratti direttamente da `df_raw`.
> - Calcola il numero complessivo di venditori unici che presentano almeno una cella corrotta, per quantificare l'impatto sul dataset prima di decidere l'azione correttiva.

In [ ]:
# Audit delle 7 colonne lette come stringa (object) a causa di punti multipli
colonne_sospette = [
    "price_median", "back_legroom_median", "front_legroom_median",
    "fuel_tank_volume_median", "height_median", "length_median", "width_median"
]

audit_sintattico = []
maschere_corrotte = {}

for col in colonne_sospette:
    serie = df_raw[col].dropna()
    # Un valore è corrotto se è presente ma non può essere convertito direttamente in numero da pandas
    numeri_test = pd.to_numeric(serie, errors="coerce")
    corrotti_mask = serie.notna() & numeri_test.isna()
    maschere_corrotte[col] = corrotti_mask
    
    n_corrotti = corrotti_mask.sum()
    pct_corrotti = (n_corrotti / len(df_raw)) * 100
    
    # Esempi reali di stringhe giganti/impossibili non manipolate dal file grezzo
    esempi_giganti = list(serie[corrotti_mask].head(3))
    esempi_str = " | ".join([f"'{x}'" for x in esempi_giganti])
    
    audit_sintattico.append({
        "Colonna": col,
        "Tipo in df_raw": str(df_raw[col].dtype),
        "N. Valori Giganti/Corrotti": n_corrotti,
        "% su Totale Dataset": f"{pct_corrotti:.2f}%",
        "Esempi di Valori Giganti Assurdi (df_raw)": esempi_str
    })

tab_audit_sintattico = pd.DataFrame(audit_sintattico)
display(tab_audit_sintattico)

# Calcolo dell'impatto complessivo a livello di venditore (righe con almeno un errore)
df_mask = pd.DataFrame(maschere_corrotte)
righe_coinvolte = df_mask.any(axis=1).sum()
tot_celle_corrotte = df_mask.sum().sum()

print("=" * 85)
print(f"RIASSUNTO DELL'AUDIT SINTATTICO SUL DATASET GREZZO:")
print(f"* Totale celle con stringhe giganti/anomale: {tot_celle_corrotte} celle")
print(f"* Totale venditori coinvolti con >= 1 anomalia: {righe_coinvolte} su {len(df_raw)} ({righe_coinvolte/len(df_raw)*100:.2f}%)")
print("=" * 85)

# Salva l'audit preliminare per documentazione nel report
from pathlib import Path
out_tab = Path('../report/tables') if Path('../report/tables').exists() else Path('report/tables')
out_tab.mkdir(parents=True, exist_ok=True)
tab_audit_sintattico.to_csv(out_tab / 'tab_audit_anomalie_sintattiche.csv', index=False)


**Commento per il report (Diagnosi del problema sintattico e motivazione dell'azione).**
- **Perché questi valori non hanno senso fisico:** valori come `'38.650.000.000.000.000'` per lo spazio per le gambe posteriore o `'7.205.000.000.000.000'` per la larghezza dell'auto equivalgono a decine di biliardi di pollici (distanze astronomiche). Similmente, un prezzo di `'26.037.495.000.000.000'` dollari supera di gran lunga l'intero PIL mondiale. Se lasciati inalterati o convertiti erroneamente, distruggerebbero completamente la media, la varianza e qualsiasi calcolo di distanza euclidea o modello supervisionato.
- **Diagnosi dell'anomalia:** non si tratta di errori di battitura casuali, bensì di un **difetto di formattazione ed esportazione software**. I numeri decimali originali sono stati salvati inserendo ripetutamente il punto come separatore delle migliaia (ad es. `38.65` è diventato `38.650.000.000.000.000`). Tutte le cifre significative sono perfettamente preservate.
- **Valutazione delle azioni possibili (Perché non eliminiamo?):**
  1. *Eliminazione delle righe:* comporterebbe la perdita di ben **1.448 venditori (il 10,46% del dataset)**. Rimuovere oltre un decimo delle osservazioni per un semplice difetto di punteggiatura creerebbe una massiccia perdita di informazione e distorsione campionaria.
  2. *Sostituzione cieca con NaN e imputazione:* trasformare 1.582 celle in valori mancanti per poi indovinarli con la mediana significherebbe buttare via l'informazione reale dei venditori quando le cifre sono già note.
  3. *Azione Adottata (Sezione 3): Riparazione sintattica deterministica.* Rimuovere i punti spuri e ricollocare il separatore decimale nell'unico intervallo fisicamente e statisticamente plausibile della colonna, recuperando oltre il 99,7% delle informazioni reali.

## 3. Riparazione dei numeri corrotti

In 7 colonne alcuni valori sono testi come `38.650.000.000.000.000` (legroom) o `6.619.999.999.999.990` (altezza): numeri decimali **rovinati da un'esportazione che ha trattato il punto come separatore delle migliaia** (38,65 → `38.650.000.000.000.000`).

Opzioni:
1. trasformarli in NaN e poi imputarli → semplice, ma si butta un'informazione che c'è;
2. **ricostruirli**: tolgo i punti, provo ogni posizione del punto decimale e tengo il candidato che cade nel range plausibile della colonna;
3. **ibrida (scelta)**: ricostruzione quando il candidato plausibile è uno solo, NaN quando ce ne sono zero o più di uno.

> **Cosa fa il codice qui sotto.** Definisce la funzione `ripara_colonna`, che riceve una colonna e restituisce la colonna riparata più l'elenco delle righe che erano corrotte. Passaggi:
> 1. `pd.to_numeric(..., errors="coerce")` converte in numeri; ciò che non si riesce a convertire (i valori corrotti) diventa NaN;
> 2. le righe corrotte sono quelle che avevano un valore (`notna`) ma dopo la conversione sono NaN (`isna`);
> 3. il range plausibile va dallo 0,1° al 99,9° percentile dei valori leggibili (`quantile`);
> 4. per ogni valore corrotto: tolgo i punti con `replace`, costruisco tutti i numeri possibili mettendo il punto decimale in ogni posizione e tengo quelli dentro il range;
> 5. se il candidato plausibile è uno solo, lo scrivo al posto del NaN (`numeri.loc[i] = ...`); altrimenti resta NaN.

In [ ]:
def ripara_colonna(colonna):
    numeri = pd.to_numeric(colonna, errors="coerce")        # i valori corrotti diventano NaN
    corrotti = colonna.notna() & numeri.isna()               # c'era un valore, ma non è leggibile
    minimo = numeri.quantile(0.001)                          # range plausibile della colonna
    massimo = numeri.quantile(0.999)

    for i in colonna[corrotti].index:
        cifre = colonna[i].replace(".", "")                  # '38.650.000...' -> '38650000...'
        # tutti i numeri ottenibili mettendo il punto in ogni posizione: 3.865, 38.65, 386.5, ...
        candidati = [float(cifre[:k] + "." + cifre[k:]) for k in range(1, len(cifre) + 1)]
        plausibili = [x for x in candidati if minimo <= x <= massimo]
        if len(plausibili) == 1:                             # un solo candidato plausibile: lo teniamo
            numeri.loc[i] = plausibili[0]                    # altrimenti resta NaN
    return numeri, corrotti

> **Cosa fa il codice qui sotto.**
> - Crea `df`, una copia di `df_raw` su cui lavoreremo (così i dati originali restano intatti).
> - Per ognuna delle 7 colonne corrotte chiama `ripara_colonna`, sostituisce la colonna con la versione riparata e conserva in `corrotti` la maschera booleana delle righe rovinate.
> - Costruisce e mostra una tabella riassuntiva (`tab_riparazione`) che conta quanti valori erano corrotti in ogni colonna, la percentuale d'impatto sul totale, il tasso di ricostruzione riuscita e due esempi concreti di valori originali alterati affiancati alla loro ricostruzione decimale plausibile.


In [ ]:
colonne_corrotte = [
    "price_median", "back_legroom_median", "front_legroom_median",
    "fuel_tank_volume_median", "height_median", "length_median", "width_median"
]

df = df_raw.copy()
corrotti = {}
dettaglio_riparazione = []

for col in colonne_corrotte:
    df[col], corrotti[col] = ripara_colonna(df_raw[col])
    idx_corrotti = corrotti[col][corrotti[col]].index
    n_corrotti = len(idx_corrotti)
    n_riparati = df.loc[idx_corrotti, col].notna().sum()
    
    # Estrae due esempi di valori originali corrotti e i corrispondenti valori riparati
    esempi_orig = list(df_raw.loc[idx_corrotti, col].head(2))
    esempi_rip = [f"{v:.2f}" for v in df.loc[idx_corrotti, col].head(2)]
    coppie_esempi = [f"{orig} -> {rip}" for orig, rip in zip(esempi_orig, esempi_rip)]
    
    dettaglio_riparazione.append({
        "Colonna": col,
        "N. Corrotti": n_corrotti,
        "% su Totale": f"{(n_corrotti / len(df_raw) * 100):.2f}%",
        "N. Ricostruiti": n_riparati,
        "% Successo": f"{(n_riparati / n_corrotti * 100 if n_corrotti > 0 else 0):.1f}%",
        "Esempi (Valore Corrotto -> Valore Ricostruito)": " | ".join(coppie_esempi)
    })

tab_riparazione = pd.DataFrame(dettaglio_riparazione)
display(tab_riparazione)

# Salva la tabella aggiornata per il report
tab_riparazione.to_csv("../report/tables/tab_riparazione_numeri_esempi.csv", index=False)


> **Cosa fa il codice qui sotto.** Controllo della riparazione per 4 colonne.
> - `plt.subplots(1, 4)` crea una figura con 4 grafici affiancati (`axes` è l'elenco dei 4 grafici).
> - In ognuno disegna due istogrammi sovrapposti: valori leggibili e valori ricostruiti. `density=True` li rende confrontabili anche se i gruppi hanno dimensioni diverse; `alpha=0.5` li rende semitrasparenti; `bins` sono gli stessi intervalli per entrambi.
> - `~corrotti[col]` vuol dire "le righe **non** corrotte".
> - `plt.savefig` salva la figura in `report/figures/` per il report.
>
> Se le due forme coincidono, la ricostruzione non ha introdotto distorsioni.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, ["back_legroom_median", "height_median", "width_median", "fuel_tank_volume_median"]):
    bins = np.histogram_bin_edges(df[col].dropna(), bins=40)      # stessi intervalli per i due istogrammi
    ax.hist(df.loc[~corrotti[col], col], bins=bins, density=True, alpha=0.5, label="leggibili")
    ax.hist(df.loc[corrotti[col], col], bins=bins, density=True, alpha=0.5, label="ricostruiti")
    ax.set_title(col)
axes[0].legend()
plt.tight_layout()
plt.savefig("../report/figures/fig03_riparazione_numeri.png", dpi=200, bbox_inches="tight")
plt.show()

> **Cosa fa il codice qui sotto.**
> - Converte `sp_id` in numero intero e lo usa come **indice** della tabella (`set_index`): diventa l'etichetta di ogni riga invece di una colonna, così non finisce per sbaglio tra le variabili delle analisi.
> - Con `replace` rinomina i codici del cambio `a`/`m` in `automatic`/`manual`.

In [ ]:
df["sp_id"] = df["sp_id"].astype(int)
df = df.set_index("sp_id")
df["dominant_transmission"] = df["dominant_transmission"].replace({"a": "automatic", "m": "manual"})

**Commento per il report.** Circa 1.580 celle corrotte; il **99,7%** è stato ricostruito in modo univoco. Gli istogrammi dei valori ricostruiti si sovrappongono a quelli dei valori leggibili, quindi la ricostruzione non introduce distorsioni. I pochi casi ambigui o senza candidato (2 prezzi, 3 misure) diventano NaN e vengono trattati in §6.

## 4. Distribuzioni e statistiche

### 4.1 Statistiche descrittive (numeriche)
Il confronto **media vs mediana** (colonna `50%`) e la **skewness** indicano l'asimmetria; `n_zeri` mette in luce le distribuzioni con molta massa in 0.

> **Cosa fa il codice qui sotto.**
> - `gruppi` è un dizionario che divide le 24 variabili numeriche in 6 famiglie (nome della famiglia → elenco di colonne): serve per fare un grafico per famiglia.
> - `numeriche` mette tutte queste colonne in un unico elenco; `categoriche` elenca le variabili categoriche. Le due liste vengono riusate in tutto il notebook.
> - `describe()` calcola per ogni colonna: conteggio, media (`mean`), deviazione standard (`std`), minimo, quartili (25%, 50% = mediana, 75%) e massimo. `.T` gira la tabella (una riga per variabile).
> - Aggiunge tre colonne: asimmetria (`skew`), numero di zeri e percentuale di NaN.

In [ ]:
gruppi = {
    "Attività del venditore": ["n_listings", "n_listing_days", "listing_span_days"],
    "Prezzo e mercato": ["price_median", "savings_amount_median", "daysonmarket_median"],
    "Età e uso": ["mileage_median", "year_median", "vehicle_age_median", "owner_count_median"],
    "Motore e consumi": ["horsepower_median", "engine_displacement_median",
                         "city_fuel_economy_median", "highway_fuel_economy_median"],
    "Dimensioni": ["back_legroom_median", "front_legroom_median", "fuel_tank_volume_median",
                   "height_median", "length_median", "width_median"],
    "Composizione inventario": ["pct_isCab", "pct_is_new", "pct_salvage", "pct_theft_title"],
}
numeriche = [col for colonne in gruppi.values() for col in colonne]
categoriche = ["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_wheel_system",
               "dominant_listing_color", "franchise_status", "new_used_profile"]

statistiche = df[numeriche].describe().T
statistiche["skew"] = df[numeriche].skew()
statistiche["n_zeri"] = (df[numeriche] == 0).sum()
statistiche["%NaN"] = df[numeriche].isna().mean() * 100
statistiche.round(2)

### 4.2 Distribuzioni univariate delle numeriche

> **Cosa fa il codice qui sotto.**
> - Per ogni famiglia di `gruppi`, `df[colonne].hist(...)` disegna un istogramma per ogni colonna della famiglia, con 40 intervalli (`bins`), su righe da 3 grafici (`layout=(-1, 3)`: il -1 vuol dire "tante righe quante servono").
> - `enumerate(..., start=1)` numera le famiglie da 1 a 6: il numero e il nome (in `nomi_file`) servono per il nome del file salvato.

In [ ]:
nomi_file = ["attivita", "mercato", "eta_uso", "motore", "dimensioni", "composizione"]

for i, (gruppo, colonne) in enumerate(gruppi.items(), start=1):
    df[colonne].hist(bins=40, layout=(-1, 3), figsize=(13, 3.2 * ((len(colonne) + 2) // 3)))
    plt.suptitle(gruppo)
    plt.tight_layout()
    plt.savefig(f"../report/figures/fig04_{i}_hist_{nomi_file[i - 1]}.png", dpi=200, bbox_inches="tight")
    plt.show()

### 4.3 Box plot

> **Cosa fa il codice qui sotto.** Con `plot(kind="box", subplots=True)` disegna un box plot separato per ogni variabile numerica, in una griglia di 4 righe × 6 colonne. La scatola va dal 1° al 3° quartile, la linea interna è la mediana, i punti oltre i baffi (1,5·IQR) sono i candidati outlier.

In [ ]:
df[numeriche].plot(kind="box", subplots=True, layout=(4, 6), figsize=(18, 10))
plt.tight_layout()
plt.savefig("../report/figures/fig04_7_boxplot_numeriche.png", dpi=200, bbox_inches="tight")
plt.show()

**Commento per il report (§4.1–4.3).**
- **Forte asimmetria positiva** (media ≫ mediana): `n_listings` (mediana 70, max 4.959), `daysonmarket_median` (max 1.987), `listing_span_days`, `price_median` (max ~236.000 USD), `savings_amount_median`. Sono distribuzioni a coda lunga, tipiche di conteggi e prezzi: negli istogrammi quasi tutti i valori sono schiacciati a sinistra. → Candidate al logaritmo (§7).
- **Asimmetria negativa** di `year_median`, per una piccola coda di concessionari di auto d'epoca (1960–1988).
- **Molta massa in 0**: `pct_salvage` (~91% zeri), `pct_theft_title` (~85%), `savings_amount_median` (~37%), `pct_is_new` (~53% = venditori di solo usato), `mileage_median` (621 venditori di solo nuovo).
- Variabili quasi costanti: `front_legroom_median` (dev. std ≈ 0,8 pollici), probabilmente poco informativa.
- I box plot segnalano molti punti oltre i baffi per le variabili asimmetriche: non sono necessariamente errori (§7.1).

### 4.4 Distribuzioni delle categoriche

> **Cosa fa il codice qui sotto.**
> - Crea una griglia di 2 × 4 grafici; `axes.ravel()` la trasforma in un elenco semplice di 8 grafici, così il ciclo `for` può riempirli uno alla volta.
> - Per ogni variabile categorica e per il target, `value_counts(dropna=False)` conta le righe per ogni valore (NaN compresi) e `plot(kind="barh")` le disegna come barre orizzontali.
> - Infine stampa la distribuzione del target in percentuale (`normalize=True` dà le proporzioni invece dei conteggi).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), categoriche + ["seller_rating_class"]):
    df[col].value_counts(dropna=False).plot(kind="barh", ax=ax, title=col)
plt.tight_layout()
plt.savefig("../report/figures/fig04_8_categoriche.png", dpi=200, bbox_inches="tight")
plt.show()

print((df["seller_rating_class"].value_counts(normalize=True) * 100).round(1))

**Commento per il report.**
- **Target sbilanciato**: medium ~55%, high ~27%, low ~19% → in classificazione serviranno split stratificato e metriche come la macro-F1, non la sola accuratezza.
- `dominant_fuel_type` (gasoline ~98,7%) e `dominant_transmission` (automatico ~94%) sono **quasi costanti**: portano pochissima informazione.
- `dominant_body_type`: SUV ~64%, sedan ~22%, pickup ~10%, le altre 6 categorie sotto il 2% → da raggruppare (§7.3).
- `dominant_wheel_system`: **76% di NaN** (§6). `dominant_listing_color = "unknown"` (~11%) è un **mancante mascherato**.
- `franchise_status` è quasi bilanciata (franchise 51%, independent 49%).

### 4.5 Bimodalità: venditori di nuovo vs di usato

> **Cosa fa il codice qui sotto.** Per chilometraggio ed età disegna un istogramma per ognuno dei tre profili nuovo/usato, sovrapposti (`hue="new_used_profile"` colora per profilo). `stat="density"` e `common_norm=False` normalizzano ogni profilo separatamente, così si confrontano le forme anche se i profili hanno numerosità molto diverse; `element="step"` disegna solo il contorno delle barre.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["mileage_median", "vehicle_age_median"]):
    sns.histplot(data=df, x=col, hue="new_used_profile", stat="density", common_norm=False,
                 element="step", bins=40, ax=ax)
plt.tight_layout()
plt.savefig("../report/figures/fig04_9_bimodalita_profilo.png", dpi=200, bbox_inches="tight")
plt.show()

### 4.6 Relazioni con il target

> **Cosa fa il codice qui sotto.**
> - `map` traduce il target in numeri: low = 0, medium = 1, high = 2. Ha senso perché il target è ordinale.
> - `corrwith(..., method="spearman")` calcola la correlazione di Spearman tra ogni variabile numerica e il target numerico.
> - `sort_values(key=abs, ascending=False)` ordina dalla correlazione più forte alla più debole, guardando il valore assoluto (così −0,26 viene prima di 0,10).
> - `piu_associate` tiene i nomi delle prime 8 variabili, usate nei grafici successivi.

In [ ]:
target_numerico = df["seller_rating_class"].map({"low": 0, "medium": 1, "high": 2})
rho_target = df[numeriche].corrwith(target_numerico, method="spearman")
rho_target = rho_target.sort_values(key=abs, ascending=False)
piu_associate = rho_target.index[:8].tolist()
rho_target.round(3)

> **Cosa fa il codice qui sotto.**
> - `coda_lunga` elenca le variabili molto asimmetriche: nei grafici le mostriamo in scala logaritmica, altrimenti le scatole sarebbero schiacciate. `np.log1p(x)` calcola log(1 + x), che funziona anche quando x = 0.
> - Per ognuna delle 8 variabili, `sns.boxplot` disegna un box plot per classe di rating (`order=classi` fissa l'ordine low → medium → high). `showfliers=False` nasconde i punti outlier per vedere meglio la parte centrale.

In [ ]:
coda_lunga = ["n_listings", "n_listing_days", "listing_span_days", "daysonmarket_median",
              "price_median", "savings_amount_median"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), piu_associate):
    if col in coda_lunga:
        valori = np.log1p(df[col])
    else:
        valori = df[col]
    sns.boxplot(x=df["seller_rating_class"], y=valori, order=classi, showfliers=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig("../report/figures/fig04_10_boxplot_vs_target.png", dpi=200, bbox_inches="tight")
plt.show()

> **Cosa fa il codice qui sotto.** Stesse 8 variabili, ma con le curve di densità (KDE, una specie di istogramma "liscio") per classe: `hue` colora per classe di rating. `common_norm=False` normalizza ogni classe separatamente: altrimenti la classe *medium*, più numerosa, schiaccerebbe le altre. Se le tre curve si sovrappongono, la variabile distingue poco le classi.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.ravel(), piu_associate):
    if col in coda_lunga:
        valori = np.log1p(df[col])
    else:
        valori = df[col]
    sns.kdeplot(x=valori, hue=df["seller_rating_class"], hue_order=classi, common_norm=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig("../report/figures/fig04_11_kde_vs_target.png", dpi=200, bbox_inches="tight")
plt.show()

> **Cosa fa il codice qui sotto.**
> - Per 4 categoriche, `pd.crosstab(..., normalize="index")` costruisce la tabella di contingenza categoria × rating e divide ogni riga per il suo totale: ogni riga diventa la distribuzione del rating dentro quella categoria, cioè P(rating | categoria). `[classi]` mette le colonne nell'ordine low, medium, high.
> - `plot(kind="barh", stacked=True)` la disegna come barre orizzontali impilate (ogni barra arriva a 1 = 100%). La legenda è mostrata solo nel primo grafico.
> - Infine stampa le tabelle per `franchise_status` e `new_used_profile`, per leggere i numeri esatti.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, col in zip(axes, ["franchise_status", "new_used_profile", "dominant_body_type", "dominant_listing_color"]):
    tabella = pd.crosstab(df[col], df["seller_rating_class"], normalize="index")[classi]
    tabella.plot(kind="barh", stacked=True, ax=ax, title=col, legend=(col == "franchise_status"))
plt.tight_layout()
plt.savefig("../report/figures/fig04_12_categoriche_vs_target.png", dpi=200, bbox_inches="tight")
plt.show()

print(pd.crosstab(df["franchise_status"], df["seller_rating_class"], normalize="index")[classi].round(3), "\n")
print(pd.crosstab(df["new_used_profile"], df["seller_rating_class"], normalize="index")[classi].round(3))

> **Cosa fa il codice qui sotto.** `df.sample(2000, random_state=42)` estrae a caso 2.000 venditori (`random_state` fissa l'estrazione, così si ottiene sempre lo stesso campione). Su questi, `sns.pairplot` disegna la scatter matrix di 4 variabili: uno scatter plot per ogni coppia e la distribuzione di ogni variabile sulla diagonale, colorati per classe. Con tutti i 13.843 punti il grafico sarebbe illeggibile e lento.

In [ ]:
campione = df.sample(2000, random_state=42)
sns.pairplot(campione, vars=["price_median", "mileage_median", "vehicle_age_median", "horsepower_median"],
             hue="seller_rating_class", hue_order=classi, plot_kws={"s": 8, "alpha": 0.4})
plt.savefig("../report/figures/fig04_13_pairplot.png", dpi=200, bbox_inches="tight")
plt.show()

> **Cosa fa il codice qui sotto.** Coordinate parallele: ogni variabile è un asse verticale, ogni venditore una linea che li attraversa.
> 1. Prende 6 variabili, togliendo le righe con NaN (`dropna`), e applica il log a quelle a coda lunga.
> 2. Le standardizza con lo z-score (`(valore − media) / dev. std`): così tutti gli assi hanno la stessa scala.
> 3. Aggiunge la colonna del target e, con `groupby(...).sample(100)`, estrae 100 venditori per classe.
> 4. `pd.plotting.parallel_coordinates` disegna una linea per venditore, colorata per classe.
> 5. Stampa la mediana di ogni variabile per classe, per leggere le differenze in numeri.

In [ ]:
colonne_pc = ["n_listings", "n_listing_days", "vehicle_age_median", "mileage_median", "pct_is_new",
              "savings_amount_median"]
pc = df[colonne_pc].dropna().copy()
pc[["n_listings", "n_listing_days", "savings_amount_median"]] = np.log1p(
    pc[["n_listings", "n_listing_days", "savings_amount_median"]])
pc = (pc - pc.mean()) / pc.std()                    # z-score: media 0, dev. std 1
pc["seller_rating_class"] = df["seller_rating_class"]

campione_pc = pc.groupby("seller_rating_class").sample(100, random_state=42)
plt.figure(figsize=(12, 5))
pd.plotting.parallel_coordinates(campione_pc, "seller_rating_class", alpha=0.25)
plt.ylim(-3, 3)
plt.savefig("../report/figures/fig04_14_coordinate_parallele.png", dpi=200, bbox_inches="tight")
plt.show()

pc.groupby("seller_rating_class").median().loc[classi].round(2)

**Commento per il report (§4.5–4.6).**
- `mileage_median` e `vehicle_age_median` sono **bimodali**: i venditori `used_only` e `mixed` formano due popolazioni distinte (km ed età alti contro bassi). Il profilo nuovo/usato è la principale fonte di struttura del dataset, da tenere a mente per il clustering.
- Le relazioni con il target sono **deboli**: la |ρ| di Spearman più alta è circa 0,26 (`n_listings`, `n_listing_days`, entrambe negative: **i venditori più grandi e attivi tendono ad avere rating più basso**). Le KDE delle tre classi si sovrappongono molto. È coerente con l'avviso delle guideline sul task di classificazione "difficile".
- `franchise_status`: gli **indipendenti** hanno più rating *high* (~33% contro ~21%) e meno *low* (~16% contro ~21%). `new_used_profile` mostra lo stesso schema (used_only ≈ independent, mixed ≈ franchise): le due variabili portano quasi la stessa informazione (§8.2). Attenzione alle categorie con pochissimi venditori (es. `new_only`, 31; colori rari): le loro percentuali sono poco affidabili.
- Nelle coordinate parallele le linee delle tre classi si mescolano completamente; le mediane per classe mostrano però tendenze coerenti: la classe *high* ha meno annunci e meno giorni di attività e auto un po' più vecchie; la classe *low* ha la quota più alta di auto nuove (venditori grandi e misti, tipicamente concessionari ufficiali).

## 5. Qualità dei dati

### 5.1 Quadro per dimensioni della qualità (DM1.01 §4.2)

| Dimensione | Domanda | Riscontro | Azione |
|---|---|---|---|
| **Accuratezza sintattica** | valori nel formato/dominio corretto? | ~1.580 numeri corrotti in 7 colonne; `sp_id` letto come decimale; `pct_*` tutte in [0, 1] | ricostruzione (§3), conversione dei tipi |
| **Accuratezza semantica** | valori plausibili nel mondo reale? | legroom 0 o > 50, consumi > 60 MPG su non elettrici, km 0 su venditori di usato | caso per caso (§5.2): errore → NaN, vero → si tiene |
| **Completezza** | quanto manca (anche mascherato)? | `wheel_system` 76%, `engine_displacement` 24,5%, colore "unknown" 11% | §6 |
| **Coerenza** | attributi coerenti tra loro e con la documentazione? | `year + age ≈ 2020`; profilo ↔ `pct_is_new` coerenti; categorie documentate assenti | §2, §5.3 |
| **Sbilanciamento** | classi e categorie equilibrate? | target 55/27/19%; gasoline 98,7%; automatico 94% | stratificazione in classificazione; raggruppamento delle categorie rare (§7.3) |
| **Attualità** (*timeliness*) | a quando risalgono i dati? | annunci intorno al 2020; 32 venditori con auto del 1960–1988 | dichiarare il periodo; le auto d'epoca sono una nicchia reale |
| **Unicità** | duplicati? | 0 righe e 0 `sp_id` duplicati | nessuna |

### 5.2 Controlli di dominio (valori sospetti)
Per ogni gruppo di valori sospetti va deciso se è un **errore** (si corregge o diventa NaN) o un **valore vero** (si tiene). È il "dilemma" visto a lezione: errore e valore vero richiedono azioni opposte.

> **Cosa fa il codice qui sotto.**
> - Ogni controllo è una **condizione** scritta sulle colonne, ad esempio `df["height_median"] > 90`: il risultato è una colonna di vero/falso, una per venditore. `&` vuol dire "e", `~` vuol dire "non", `isin([...])` controlla se il valore è in un elenco.
> - I controlli sono raccolti in un dizionario (nome → condizione). Per ognuno, `.sum()` conta quanti venditori lo soddisfano (i "vero" valgono 1).

In [ ]:
elettrico = df["dominant_fuel_type"] == "electric"
ibrido_o_elettrico = df["dominant_fuel_type"].isin(["electric", "hybrid"])
consumo_urbano = df["city_fuel_economy_median"]
consumo_extraurbano = df["highway_fuel_economy_median"]

controlli = {
    "back_legroom_median = 0": df["back_legroom_median"] == 0,
    "front_legroom_median > 50": df["front_legroom_median"] > 50,
    "height_median > 90": df["height_median"] > 90,
    "consumo urbano > 60 MPG, elettrici": (consumo_urbano > 60) & elettrico,
    "consumo urbano > 60 MPG, non elettrici": (consumo_urbano > 60) & ~elettrico,
    "extraurbano < urbano": consumo_extraurbano < consumo_urbano,
    "extraurbano < urbano, ibridi o elettrici": (consumo_extraurbano < consumo_urbano) & ibrido_o_elettrico,
    "year_median < 1990": df["year_median"] < 1990,
    "daysonmarket_median > 365": df["daysonmarket_median"] > 365,
    "listing_span_days > 1500": df["listing_span_days"] > 1500,
    "mileage = 0 ma pct_is_new < 0.5": (df["mileage_median"] == 0) & (df["pct_is_new"] < 0.5),
}
for nome, condizione in controlli.items():
    print(f"{nome}: {condizione.sum()} venditori")

| Controllo | Classificazione | Azione |
|---|---|---|
| `back_legroom_median = 0` | dubbio (pickup a cabina singola?) | → NaN ("non applicabile") |
| `front_legroom_median > 50` | errore (fuori scala) | → NaN |
| `height_median > 90` | vero (van alti) | si tengono |
| consumo > 60 MPG, elettrici | vero (unità MPGe) | si tengono |
| consumo > 60 MPG, non elettrici | dubbio (ibridi mal etichettati?) | → NaN |
| extraurbano < urbano | vero (tipico di ibridi ed elettrici) | si tengono |
| `year_median < 1990` | vero (auto d'epoca) | si tengono |
| `daysonmarket_median > 365`, `listing_span_days > 1500` | vero (coda lunga) | log (§7) |
| `mileage = 0` ma `pct_is_new < 0.5` | errore (incoerente con un inventario di usato) | → NaN |

### 5.3 Coerenza tra variabili
Anche i controlli superati vanno riportati: mostrano che la verifica è stata fatta.

> Attenzione: il controllo `n_listing_days ≤ listing_span_days` sarebbe **sbagliato**. Un venditore con annunci in un solo giorno ha 1 giorno distinto e span 0. La condizione corretta è `listing_span_days ≥ n_listing_days − 1`.

> **Cosa fa il codice qui sotto.**
> - Come sopra, ogni regola è una condizione; qui però la condizione descrive la **violazione**, quindi `.sum()` conta i venditori che **non** rispettano la regola (0 = regola rispettata da tutti). L'ultima è la regola sbagliata, per mostrare quanti falsi errori darebbe.
> - Infine conta quanti venditori hanno ogni valore di `year + age` (arrotondato con `round`), per spiegare l'unico controllo con eccezioni.

In [ ]:
violazioni = {
    "listing_span_days < n_listing_days − 1": df["listing_span_days"] < df["n_listing_days"] - 1,
    "n_listing_days > n_listings": df["n_listing_days"] > df["n_listings"],
    "used_only ma pct_is_new ≠ 0": (df["new_used_profile"] == "used_only") & (df["pct_is_new"] != 0),
    "new_only ma pct_is_new ≠ 1": (df["new_used_profile"] == "new_only") & (df["pct_is_new"] != 1),
    "used_only ma mileage = 0": (df["new_used_profile"] == "used_only") & (df["mileage_median"] == 0),
    "|year + age − 2020| > 1": (df["year_median"] + df["vehicle_age_median"] - 2020).abs() > 1,
    "regola SBAGLIATA: n_listing_days > listing_span_days": df["n_listing_days"] > df["listing_span_days"],
}
for nome, condizione in violazioni.items():
    print(f"{nome}: {condizione.sum()}")

(df["year_median"] + df["vehicle_age_median"]).round().value_counts().sort_index()

> **Cosa fa il codice qui sotto.** Applica le decisioni della tabella sopra. `df.loc[condizione, colonna] = np.nan` vuol dire: "nelle righe dove la condizione è vera, scrivi NaN in quella colonna". Le righe restano; i NaN verranno imputati nella sezione 6.

In [ ]:
df.loc[df["back_legroom_median"] == 0, "back_legroom_median"] = np.nan
df.loc[df["front_legroom_median"] > 50, "front_legroom_median"] = np.nan
consumo_sospetto = (df["city_fuel_economy_median"] > 60) & (df["dominant_fuel_type"] != "electric")
df.loc[consumo_sospetto, ["city_fuel_economy_median", "highway_fuel_economy_median"]] = np.nan
df.loc[(df["mileage_median"] == 0) & (df["pct_is_new"] < 0.5), "mileage_median"] = np.nan

**Commento per il report.** Le anomalie semantiche sono **poche** (decine di venditori su 13.843) e quasi tutte spiegabili: auto elettriche con consumi in MPGe, van alti, auto d'epoca e ibridi con consumo extraurbano peggiore di quello urbano sono **valori veri** e si tengono. Diventano NaN solo i valori fisicamente impossibili o incoerenti (legroom fuori scala, km nulli in inventari di usato) e i casi dubbi (2 "gasoline" a ~72 MPG).

I controlli di coerenza sono tutti superati tranne uno: per circa l'1,7% dei venditori `year + age` si allontana da 2020 di più di un anno (valori tra 2015 e 2024, quasi tutti 2019 o 2021–2022). **Non è un errore**: la somma di due mediane non coincide con la mediana della somma, e gli annunci di un venditore coprono più anni (`listing_span_days` arriva a ~10 anni).

## 6. Valori mancanti

### 6.1 Quanti sono e come si distribuiscono

> **Cosa fa il codice qui sotto.**
> - Costruisce una tabella con numero (`isna().sum()`) e percentuale (`isna().mean() * 100`) di NaN per colonna, e stampa solo le colonne con almeno un NaN, dalla più alla meno colpita.
> - Per ogni categorica conta i valori `"unknown"`: sono mancanti "mascherati", che pandas non riconosce come NaN.

In [ ]:
mancanti = pd.DataFrame({"n_NaN": df.isna().sum(), "%NaN": (df.isna().mean() * 100).round(2)})
print(mancanti[mancanti["n_NaN"] > 0].sort_values("n_NaN", ascending=False), "\n")

for col in categoriche:
    n_unknown = (df[col] == "unknown").sum()
    if n_unknown > 0:
        print(f"'unknown' in {col}: {n_unknown}")

> **Cosa fa il codice qui sotto.**
> 1. Prende solo le colonne che hanno almeno un NaN.
> 2. Ordina i venditori dal più "incompleto" al meno (numero di NaN per riga, `sum(axis=1)`).
> 3. `sns.heatmap` disegna la mappa: ogni riga del grafico è una colonna del dataset, ogni punto un venditore, chiaro dove il valore manca. Con l'ordinamento, se i mancanti sono "in blocco" (stesse righe in più colonne) si vede subito.
> 4. Esamina i 27 venditori che hanno `pct_isCab` mancante: in quali altre colonne manca un valore e che profilo nuovo/usato hanno.

In [ ]:
colonne_con_nan = df.columns[df.isna().any()]
ordine_righe = df[colonne_con_nan].isna().sum(axis=1).sort_values(ascending=False).index

plt.figure(figsize=(12, 4))
sns.heatmap(df.loc[ordine_righe, colonne_con_nan].isna().T, cbar=False, xticklabels=False)
plt.title("Mappa dei valori mancanti")
plt.savefig("../report/figures/fig06_1_mappa_mancanti.png", dpi=200, bbox_inches="tight")
plt.show()

blocco = df["pct_isCab"].isna()
print("Venditori con NaN in blocco:", blocco.sum())
print(df.loc[blocco].isna().sum()[lambda s: s > 0], "\n")
print(df.loc[blocco, "new_used_profile"].value_counts())

### 6.2 Il fatto di mancare porta informazione?
Per le due variabili con molti mancanti si confrontano le righe con e senza mancante: mediane delle altre variabili e distribuzione del target, con la **V di Cramér** (da 0 = indipendenza a 1 = associazione perfetta). Il target viene usato **solo per descrivere**, mai per imputare.

> **Cosa fa il codice qui sotto.**
> - Definisce `cramers_v`: costruisce la tabella di contingenza tra due variabili, ne calcola il χ² con `chi2_contingency` e lo normalizza con la formula V = √(χ² / (n · (min(righe, colonne) − 1))).
> - Per le due variabili, `manca` è una colonna vero/falso (il valore manca o no). `groupby(manca)` divide i venditori nei due gruppi e calcola le mediane di 4 variabili in ciascuno; `crosstab` confronta la distribuzione del rating nei due gruppi; infine stampa la V di Cramér tra "manca" e il target.

In [ ]:
def cramers_v(x, y):
    tabella = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabella, correction=False)[0]
    n = tabella.values.sum()
    return np.sqrt(chi2 / (n * (min(tabella.shape) - 1)))


for col in ["engine_displacement_median", "dominant_wheel_system"]:
    manca = df[col].isna().rename("manca")
    print(f"=== {col}: {manca.mean():.1%} mancanti ===")
    print(df.groupby(manca)[["n_listings", "price_median", "vehicle_age_median", "horsepower_median"]].median())
    print(pd.crosstab(manca, df["seller_rating_class"], normalize="index")[classi].round(3))
    print("V di Cramér con il target:", round(cramers_v(manca, df["seller_rating_class"]), 3), "\n")

**Come leggere il risultato.** Se le mediane e la distribuzione del target sono quasi uguali tra i due gruppi (V di Cramér ≈ 0), i mancanti sono "casuali" e si possono imputare o eliminare senza problemi. Se invece differiscono, il mancante è **informativo**: conviene tenerne traccia con una colonna 0/1 o con una categoria `unknown`.

### 6.3 Mancanti strutturali: "non applicabile"
I 27 venditori con NaN in blocco su `pct_isCab`, `pct_salvage`, `pct_theft_title` e `owner_count_median` sono **quasi tutti venditori di solo nuovo** (26 dei 31 `new_only` del dataset). Per un'auto nuova queste grandezze **non si applicano**: non ha proprietari precedenti e non può essere un ex taxi, un'incidentata o un'auto rubata.

Scelta:
- **non eliminare le righe**: si perderebbe l'84% della categoria `new_only`;
- `pct_isCab`, `pct_salvage`, `pct_theft_title` → **0**;
- `owner_count_median` → **0** per i venditori con almeno il 90% di auto nuove (0 proprietari precedenti).

> **Cosa fa il codice qui sotto.**
> - `drop(columns=...)` elimina la colonna `dominant_wheel_system` (76% NaN).
> - `fillna(0)` sostituisce i NaN con 0 nelle tre colonne `pct_*`.
> - Per `owner_count_median` mette 0 solo nelle righe dove il valore manca **e** le auto nuove sono almeno il 90%; gli altri pochi NaN verranno imputati nel passo successivo.

In [ ]:
df = df.drop(columns="dominant_wheel_system")

for col in ["pct_isCab", "pct_salvage", "pct_theft_title"]:
    df[col] = df[col].fillna(0)

quasi_tutto_nuovo = df["owner_count_median"].isna() & (df["pct_is_new"] >= 0.9)
df.loc[quasi_tutto_nuovo, "owner_count_median"] = 0
print("owner_count_median posti a 0:", quasi_tutto_nuovo.sum())

### 6.4 Imputazione dei pochi mancanti residui
Le variabili con meno dell'1% di NaN si imputano con la **mediana del gruppo `dominant_body_type`**: dimensioni e consumi dipendono dal tipo di carrozzeria, e la mediana è robusta agli outlier.

> **Cosa fa il codice qui sotto.**
> - `pochi_nan` elenca le numeriche che hanno ancora qualche NaN, esclusa la cilindrata (trattata a parte al punto 6.5).
> - Per ogni colonna, `groupby("dominant_body_type")[col].transform("median")` calcola la mediana del gruppo di carrozzeria e la "riporta" su ogni riga: ogni venditore riceve la mediana del suo gruppo (es. la mediana dei pickup).
> - `fillna(...)` usa quel valore solo dove c'è un NaN.
> - Infine verifica che in queste colonne non sia rimasto nessun NaN.

In [ ]:
pochi_nan = [c for c in numeriche if c != "engine_displacement_median" and df[c].isna().any()]
print("Colonne imputate:", pochi_nan)

for col in pochi_nan:
    mediana_del_gruppo = df.groupby("dominant_body_type")[col].transform("median")
    df[col] = df[col].fillna(mediana_del_gruppo)

print("NaN rimasti in queste colonne:", df[pochi_nan].isna().sum().sum())

### 6.5 `engine_displacement_median` (24,5% di NaN): confronto di 4 strategie
Come nel lab Titanic con `Age`, si confrontano 4 strategie: mediana globale, mediana per carrozzeria, estrazione casuale dai valori osservati (DM1.02 §8), regressione lineare da potenza e serbatoio.

Per misurare quale funziona meglio si **nasconde il 20% dei valori noti**, lo si ricostruisce con ogni strategia (stimata solo sull'altro 80%) e si misura l'errore medio (MAE).

> **Cosa fa il codice qui sotto.**
> 1. `noti` sono i venditori con la cilindrata presente. `sample(frac=0.2)` ne estrae a caso il 20% (`test`, i valori "nascosti" da indovinare); il resto è `train`.
> 2. Calcola le stime delle 4 strategie per le righe di `test`, usando solo `train`:
>    - mediana globale: lo stesso valore per tutti;
>    - mediana per carrozzeria: `map` assegna a ogni venditore la mediana del suo tipo di carrozzeria;
>    - estrazione casuale: valori pescati a caso tra quelli di `train`;
>    - regressione: `LinearRegression().fit(X, y)` impara una formula `cilindrata ≈ a·potenza + b·serbatoio + c` e `predict` la applica.
> 3. Per ogni strategia stampa l'errore medio (media delle differenze in valore assoluto tra stima e valore vero) e la deviazione standard delle stime, da confrontare con quella dei valori veri.

In [ ]:
predittori = ["horsepower_median", "fuel_tank_volume_median"]

noti = df[df["engine_displacement_median"].notna()]
test = noti.sample(frac=0.2, random_state=42)       # 20% dei valori noti, che "nascondiamo"
train = noti.drop(test.index)
valori_veri = test["engine_displacement_median"]

mediana_per_carrozzeria = train.groupby("dominant_body_type")["engine_displacement_median"].median()
estratti = train["engine_displacement_median"].sample(len(test), replace=True, random_state=42)
regressione = LinearRegression().fit(train[predittori], train["engine_displacement_median"])

stime = {
    "mediana globale": pd.Series(train["engine_displacement_median"].median(), index=test.index),
    "mediana per carrozzeria": test["dominant_body_type"].map(mediana_per_carrozzeria),
    "estrazione casuale": pd.Series(estratti.values, index=test.index),
    "regressione": pd.Series(regressione.predict(test[predittori]), index=test.index),
}

for nome, stima in stime.items():
    errore = (stima - valori_veri).abs().mean()
    print(f"{nome}: errore medio = {errore:.0f} cc | dev. std delle stime = {stima.std():.0f}")
print(f"(dev. std dei valori veri = {valori_veri.std():.0f})")

> **Cosa fa il codice qui sotto.** Per ogni strategia sovrappone l'istogramma dei valori veri nascosti e quello delle stime. Una buona strategia produce stime con una forma simile ai valori veri; la mediana globale, invece, produce un'unica barra altissima (tutte le stime uguali).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharey=True)
bins = np.histogram_bin_edges(noti["engine_displacement_median"], bins=40)
for ax, (nome, stima) in zip(axes, stime.items()):
    ax.hist(valori_veri, bins=bins, alpha=0.5, label="valori veri")
    ax.hist(stima, bins=bins, alpha=0.5, label="stime")
    ax.set_title(nome)
axes[0].legend()
plt.tight_layout()
plt.savefig("../report/figures/fig06_2_imputazione_cilindrata.png", dpi=200, bbox_inches="tight")
plt.show()

> **Cosa fa il codice qui sotto.**
> - Crea la colonna `engine_displacement_missing`: 1 se la cilindrata mancava (quindi sarà stimata), 0 altrimenti. Così si sa sempre quali valori sono stimati.
> - Rifà la regressione usando **tutti** i valori noti e la usa per riempire i veri mancanti.
> - Sostituisce i pochi `"unknown"` di alimentazione e cambio con il valore più frequente (la moda).
> - Controlla che nel dataset non resti nessun NaN.

In [ ]:
manca_cilindrata = df["engine_displacement_median"].isna()
df["engine_displacement_missing"] = manca_cilindrata.astype(int)

regressione = LinearRegression().fit(noti[predittori], noti["engine_displacement_median"])
df.loc[manca_cilindrata, "engine_displacement_median"] = regressione.predict(df.loc[manca_cilindrata, predittori])

df["dominant_fuel_type"] = df["dominant_fuel_type"].replace("unknown", "gasoline")          # moda
df["dominant_transmission"] = df["dominant_transmission"].replace("unknown", "automatic")    # moda

print("NaN rimasti nel dataset:", df.isna().sum().sum())

**Commento per il report.**
- **Il fatto di mancare non porta informazione** né per `engine_displacement_median` né per `dominant_wheel_system`: le mediane delle altre variabili e la distribuzione del rating sono praticamente uguali tra righe con e senza mancante (V di Cramér con il target ≈ 0,01). I mancanti sono quindi "casuali" (un problema di raccolta, non una caratteristica dei venditori).
- `dominant_wheel_system` (76% NaN) è stata **eliminata**: imputare tre quarti dei valori vorrebbe dire inventare la variabile.
- I 27 NaN **in blocco** sono **mancanti strutturali** di venditori di solo nuovo ("non applicabile"): sono stati posti a 0 invece di eliminare le righe.
- `engine_displacement_median` (24,5% NaN): la **regressione** su potenza e serbatoio ricostruisce i valori nascosti con l'errore più basso (≈ 330 cc contro ≈ 620 della mediana globale e ≈ 520 della mediana per carrozzeria). La mediana globale mette lo stesso valore ovunque (un picco artificiale, la stessa distorsione vista nel lab Titanic); l'estrazione casuale rispetta la forma della distribuzione ma sbaglia di più, perché ignora le caratteristiche del venditore.
- I pochi `unknown` di alimentazione e cambio sono stati sostituiti con la moda; il colore `unknown` (~11%) resta una categoria, perché non c'è un modo sensato di ricostruirlo.
- **Data leakage**: qui le imputazioni sono stimate su tutto il dataset, il che va bene per l'esplorazione. In classificazione andranno ristimate sul solo training set.

## 7. Outlier, trasformazioni e nuove variabili

### 7.1 Outlier univariati (regola k·IQR)

> **Cosa fa il codice qui sotto.**
> - Definisce `quota_outlier`: calcola il 1° e il 3° quartile (`quantile([0.25, 0.75])`), l'IQR = Q3 − Q1, e restituisce la percentuale di valori fuori dai recinti [Q1 − k·IQR, Q3 + k·IQR]. `k=1.5` è il valore usato se non se ne indica un altro.
> - La applica a ogni variabile numerica con k = 1,5 (outlier) e k = 3 (outlier estremi) e stampa la tabella ordinata.
> - `nlargest(5, "n_listings")` mostra i 5 venditori con più annunci.

In [ ]:
def quota_outlier(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    fuori = (serie < q1 - k * iqr) | (serie > q3 + k * iqr)
    return fuori.mean() * 100


outlier = pd.DataFrame({
    "% outlier 1.5·IQR": [quota_outlier(df[c], 1.5) for c in numeriche],
    "% outlier 3·IQR": [quota_outlier(df[c], 3) for c in numeriche],
}, index=numeriche)
print(outlier.round(2).sort_values("% outlier 1.5·IQR", ascending=False))

df.nlargest(5, "n_listings")[["n_listings", "n_listing_days", "franchise_status", "pct_is_new", "seller_rating_class"]]

**Commento.**
- Per le variabili molto asimmetriche la regola 1,5·IQR segnala centinaia di "outlier" (es. `n_listings` ~4,6%, `daysonmarket_median` ~8%). Non sono errori: è la **coda di una distribuzione a coda lunga**, e venditori molto grandi o auto d'epoca esistono davvero. La scelta è **tenerli e ridurne l'effetto con il logaritmo** (§7.2).
- **Limite della regola IQR**: per `pct_salvage` e `pct_theft_title` più del 75% dei valori è 0, quindi Q1 = Q3 = 0 e IQR = 0. **Ogni valore diverso da zero** risulta "outlier", e la regola non ha senso. Per queste variabili si binarizza (§7.3).
- Da ricordare per il Modulo 2: K-means è sensibile agli outlier, DBSCAN li potrà etichettare come rumore.

### 7.2 Trasformazione logaritmica
`log1p(x) = log(1 + x)` comprime la coda destra e funziona anche con gli zeri. Si applica a **conteggi, durate e importi** molto asimmetrici. Per confronto si riporta anche la **radice quadrata**, una trasformazione più "morbida".

> **Cosa fa il codice qui sotto.** Per 5 variabili a coda lunga costruisce una tabella con: asimmetria prima, dopo il logaritmo (`np.log1p`) e dopo la radice quadrata (`np.sqrt`); percentuale di outlier prima e dopo il logaritmo.

In [ ]:
da_trasformare = ["n_listings", "listing_span_days", "daysonmarket_median", "price_median", "savings_amount_median"]

pd.DataFrame({
    "skew prima": df[da_trasformare].skew(),
    "skew dopo log1p": np.log1p(df[da_trasformare]).skew(),
    "skew dopo sqrt": np.sqrt(df[da_trasformare]).skew(),
    "% outlier prima": [quota_outlier(df[c]) for c in da_trasformare],
    "% outlier dopo log1p": [quota_outlier(np.log1p(df[c])) for c in da_trasformare],
}, index=da_trasformare).round(2)

> **Cosa fa il codice qui sotto.** Per 4 variabili disegna l'istogramma originale (riga sopra) e quello dopo il logaritmo (riga sotto). `axes[0, j]` è il grafico in riga 0 e colonna j.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 6))
for j, col in enumerate(["n_listings", "daysonmarket_median", "price_median", "listing_span_days"]):
    df[col].hist(bins=50, ax=axes[0, j])
    axes[0, j].set_title(col)
    np.log1p(df[col]).hist(bins=50, ax=axes[1, j])
    axes[1, j].set_title(f"log1p({col})")
plt.tight_layout()
plt.savefig("../report/figures/fig07_1_log_prima_dopo.png", dpi=200, bbox_inches="tight")
plt.show()

**Commento per il report.** Il logaritmo porta l'asimmetria vicino a 0 per `n_listings`, `price_median`, `daysonmarket_median` e `savings_amount_median`, e riduce molto la quota di outlier IQR (per `n_listings` da ~4,6% a quasi 0). Due limiti da dire:
- per `listing_span_days` il log **corregge troppo** (asimmetria da positiva a negativa) e la radice quadrata dà un'asimmetria più vicina a 0;
- per `savings_amount_median` il ~37% di zeri resta un picco separato anche dopo il log.

Per semplicità e coerenza si usa `log1p` per tutte (è monotona, quindi non cambia l'ordine dei valori). La radice quadrata per `listing_span_days` resta un'alternativa da provare nel clustering.

### 7.3 Categorie rare e binarizzazione
- `dominant_body_type`: le 6 categorie sotto il 2% vengono raggruppate: `coupe_convertible` (sportive) e `other` (van, minivan, hatchback, wagon).
- `dominant_listing_color`: brown, green, gold e teal (meno dello 0,3% insieme) → `other`.
- `pct_salvage` e `pct_theft_title` sono 0 per l'85–91% dei venditori → diventano binarie (`has_salvage`, `has_theft_title`): l'informazione utile è "ne ha o no".

> **Cosa fa il codice qui sotto.**
> - `replace({vecchio: nuovo, ...})` rinomina le categorie rare.
> - `(df["pct_salvage"] > 0)` dà vero/falso per ogni venditore; `.astype(int)` lo trasforma in 1/0.
> - Stampa le nuove frequenze delle carrozzerie e la quota di venditori con valore 1 nelle due variabili binarie.

In [ ]:
df["dominant_body_type"] = df["dominant_body_type"].replace({
    "coupe": "coupe_convertible", "convertible": "coupe_convertible",
    "van": "other", "minivan": "other", "hatchback": "other", "wagon": "other",
})
df["dominant_listing_color"] = df["dominant_listing_color"].replace(
    {"brown": "other", "green": "other", "gold": "other", "teal": "other"})

df["has_salvage"] = (df["pct_salvage"] > 0).astype(int)
df["has_theft_title"] = (df["pct_theft_title"] > 0).astype(int)

print(df["dominant_body_type"].value_counts(normalize=True).round(3))
print(df[["has_salvage", "has_theft_title"]].mean().round(3))

### 7.4 Nuove variabili (feature engineering)
| Nuova variabile | Definizione | Perché |
|---|---|---|
| `listings_per_day` | `n_listings / n_listing_days` | intensità dell'attività; sostituisce una coppia ridondante (ρ ≈ 0,96) |
| `combined_fuel_economy` | `0.55·city + 0.45·highway` (pesi EPA) | sostituisce due variabili con ρ ≈ 0,94 |
| `discount_ratio` | `savings_amount / price` | sconto relativo, confrontabile tra fasce di prezzo |

> **Cosa fa il codice qui sotto.** Crea le 3 colonne con operazioni tra colonne (fatte riga per riga), ne calcola la correlazione di Spearman con il target e disegna un box plot per classe di rating per ognuna.

In [ ]:
df["listings_per_day"] = df["n_listings"] / df["n_listing_days"]
df["combined_fuel_economy"] = 0.55 * df["city_fuel_economy_median"] + 0.45 * df["highway_fuel_economy_median"]
df["discount_ratio"] = df["savings_amount_median"] / df["price_median"]
nuove = ["listings_per_day", "combined_fuel_economy", "discount_ratio"]

print(df[nuove].corrwith(target_numerico, method="spearman").round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, nuove):
    sns.boxplot(x=df["seller_rating_class"], y=df[col], order=classi, showfliers=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig("../report/figures/fig07_2_nuove_feature.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Correlazioni ed eliminazione di variabili

### 8.1 Matrice di correlazione
**Spearman** è la misura principale (relazioni monotone, robusta a outlier e asimmetrie); **Pearson** serve per confronto: dove i due valori differiscono molto, la relazione non è lineare oppure è guidata da outlier.

> **Cosa fa il codice qui sotto.** `corr(method=...)` calcola la correlazione tra ogni coppia di colonne numeriche (nuove variabili comprese), con i due metodi. `sns.heatmap` disegna la matrice di Spearman come una griglia colorata: rosso = correlazione positiva, blu = negativa, bianco ≈ 0.

In [ ]:
spearman = df[numeriche + nuove].corr(method="spearman")
pearson = df[numeriche + nuove].corr(method="pearson")

plt.figure(figsize=(14, 11))
sns.heatmap(spearman, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlazione di Spearman")
plt.savefig("../report/figures/fig08_1_heatmap_spearman.png", dpi=200, bbox_inches="tight")
plt.show()

> **Cosa fa il codice qui sotto.**
> - Due cicli `for` annidati prendono ogni coppia di variabili una sola volta (la seconda variabile parte sempre dopo la prima) e salvano Spearman e Pearson della coppia.
> - Stampa le coppie con |ρ di Spearman| ≥ 0,7 (correlazione forte), dalla più forte.
> - Mostra le 8 coppie in cui Spearman e Pearson differiscono di più.

In [ ]:
colonne_corr = numeriche + nuove
coppie = []
for i, a in enumerate(colonne_corr):
    for b in colonne_corr[i + 1:]:
        coppie.append((a, b, spearman.loc[a, b], pearson.loc[a, b]))
coppie = pd.DataFrame(coppie, columns=["var_1", "var_2", "spearman", "pearson"])
coppie["differenza"] = (coppie["spearman"] - coppie["pearson"]).abs()

print(coppie[coppie["spearman"].abs() >= 0.7].sort_values("spearman", key=abs, ascending=False).round(3))
coppie.sort_values("differenza", ascending=False).head(8).round(3)

### 8.2 Associazione tra categoriche (V di Cramér)
Per le categoriche la correlazione non è definita: si usa la V di Cramér (definita al §6.2).

> **Cosa fa il codice qui sotto.** Crea una tabella vuota con le categoriche (target compreso) sia sulle righe sia sulle colonne, la riempie con la V di Cramér di ogni coppia (due cicli `for`) e la disegna come heatmap con i valori scritti nelle celle (`annot=True`).

In [ ]:
categoriche_ora = ["dominant_body_type", "dominant_fuel_type", "dominant_transmission", "dominant_listing_color",
                   "franchise_status", "new_used_profile", "has_salvage", "has_theft_title", "seller_rating_class"]

v_cramer = pd.DataFrame(index=categoriche_ora, columns=categoriche_ora, dtype=float)
for a in categoriche_ora:
    for b in categoriche_ora:
        v_cramer.loc[a, b] = cramers_v(df[a], df[b])

plt.figure(figsize=(8, 6.5))
sns.heatmap(v_cramer, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues")
plt.title("V di Cramér tra variabili categoriche")
plt.savefig("../report/figures/fig08_2_cramers_v.png", dpi=200, bbox_inches="tight")
plt.show()

### 8.3 Alternativa: PCA sul blocco "dimensioni del veicolo"
Dimensioni, serbatoio, potenza e cilindrata sono correlate tra loro. Invece di eliminarne alcune, si possono riassumere con una PCA (DM1.02 §5): la ridondanza si riduce senza scegliere a mano cosa scartare, ma le componenti sono meno interpretabili.

> **Cosa fa il codice qui sotto.**
> - `StandardScaler().fit_transform(...)` standardizza le 6 variabili (media 0, dev. std 1): la PCA lo richiede, altrimenti dominerebbero le variabili con valori grandi.
> - `PCA().fit(...)` calcola le componenti principali. `explained_variance_ratio_` è la quota di varianza spiegata da ogni componente; `cumsum()` la somma progressiva.
> - Disegna le quote come barre e mostra i pesi (`components_`) delle 6 variabili nelle prime due componenti.

In [ ]:
dimensioni = ["length_median", "width_median", "height_median", "fuel_tank_volume_median",
              "horsepower_median", "engine_displacement_median"]

pca = PCA().fit(StandardScaler().fit_transform(df[dimensioni]))
print("Varianza spiegata:", pca.explained_variance_ratio_.round(3))
print("Cumulata:         ", pca.explained_variance_ratio_.cumsum().round(3))

plt.bar(range(1, 7), pca.explained_variance_ratio_)
plt.xlabel("componente principale")
plt.ylabel("quota di varianza spiegata")
plt.savefig("../report/figures/fig08_3_pca_dimensioni.png", dpi=200, bbox_inches="tight")
plt.show()

pd.DataFrame(pca.components_[:2].T, index=dimensioni, columns=["PC1", "PC2"]).round(3)

### 8.4 Regola di decisione e variabili eliminate
**Regola dichiarata in anticipo**:
- |ρ| ≥ 0,9 → si tiene una sola variabile (la più interpretabile, con meno valori imputati, più legata al target) oppure si fondono le due in una nuova variabile;
- 0,7 ≤ |ρ| < 0,9 → caso per caso (spesso si tengono entrambe: sono correlate ma non duplicate);
- variabili **quasi costanti** o **derivate** da altre → escluse.

| Variabile | Decisione | Motivo |
|---|---|---|
| `year_median` | eliminata | ρ ≈ −0,98 con `vehicle_age_median` (età = 2020 − anno) |
| `n_listing_days` | eliminata | ρ ≈ 0,96 con `n_listings`; l'informazione resta in `listings_per_day` |
| `city_/highway_fuel_economy_median` | eliminate | ρ ≈ 0,94 tra loro; sostituite da `combined_fuel_economy` |
| `pct_salvage`, `pct_theft_title` | eliminate | quasi sempre 0; sostituite dalle versioni binarie |
| `dominant_fuel_type` | eliminata | quasi costante (~99% gasoline) |
| `dominant_wheel_system` | eliminata (§6) | 76% di NaN |
| `new_used_profile` | tenuta, esclusa dai modelli | determinata da `pct_is_new`; utile per il pattern mining |
| `engine_displacement_median` | tenuta, esclusa dal clustering | 24,5% stimato e ridondante con potenza e serbatoio |
| `savings_amount_median` | tenuta, esclusa dal clustering | ρ ≈ 0,91 con `discount_ratio` |
| `dominant_transmission` | tenuta solo nel dataset pulito | quasi costante (~94% automatico) |

**Commento per il report (§8).**
- Le correlazioni forti formano **blocchi interpretabili**: (i) *età/uso/nuovo* (`vehicle_age`, `year`, `mileage`, `owner_count`, `pct_is_new`, `savings`); (ii) *dimensione del veicolo* (dimensioni, serbatoio, potenza, cilindrata, consumi con segno negativo); (iii) *attività del venditore* (`n_listings`, `n_listing_days`).
- In diverse coppie Spearman è molto più alto di Pearson (es. `price`–`year`: 0,82 contro 0,55): la relazione è **monotona ma non lineare** e risente della coda di prezzi alti. È un'ulteriore ragione per usare il log su prezzo e conteggi.
- Anche le nuove variabili vanno controllate con la stessa regola: `discount_ratio` ha ρ ≈ 0,91 con `savings_amount_median`, quindi nel clustering si tiene solo `discount_ratio` (relativo al prezzo, confrontabile tra venditori).
- Tra le categoriche, `franchise_status` e `new_used_profile` sono fortemente associate (V ≈ 0,9: i concessionari ufficiali vendono anche nuovo, gli indipendenti quasi solo usato). Tutte le categoriche hanno un'associazione debole con il target (V al massimo ≈ 0,14).
- La PCA sul blocco dimensioni/motore spiega il **71%** della varianza con la sola PC1, che ha pesi simili e positivi su tutte e sei le variabili ("taglia/potenza del veicolo"). È un'alternativa da valutare nel clustering.

## 9. Dataset finali e riepilogo delle decisioni

Si salvano due file in `data/processed/` (stesse righe, indice `sp_id`):
1. **`seller_clean.csv`**: dataset ripulito, **non trasformato né scalato**, con target. Base per classificazione, regressione e (dopo la discretizzazione) pattern mining.
2. **`seller_clustering_base.csv`**: solo variabili numeriche non ridondanti, log dove serve, **z-score**, **senza target** (il target serve solo dopo, per interpretare i cluster). La scelta finale degli attributi per il clustering si fa nel Modulo 2.

> **Cosa fa il codice qui sotto.**
> 1. `df_clean` è `df` senza le variabili eliminate (elenco in `eliminate`).
> 2. `df_cluster` prende da `df_clean` le 19 colonne per il clustering; applica `np.log1p` alle 5 a coda lunga; poi `StandardScaler().fit_transform` porta ogni colonna a media 0 e dev. std 1 (il risultato è una tabella di numeri senza nomi, quindi `pd.DataFrame(...)` rimette indice e nomi delle colonne). Le colonne trasformate col log vengono rinominate con il prefisso `log1p_`.
> 3. `assert` ferma il notebook con un errore se la condizione è falsa: qui controlla che non ci siano NaN.
> 4. `to_csv` salva i due file.

In [ ]:
eliminate = ["year_median", "n_listing_days", "city_fuel_economy_median", "highway_fuel_economy_median",
             "pct_salvage", "pct_theft_title", "dominant_fuel_type"]
df_clean = df.drop(columns=eliminate)

colonne_cluster = ["n_listings", "listings_per_day", "listing_span_days", "daysonmarket_median",
                   "price_median", "discount_ratio", "mileage_median", "vehicle_age_median",
                   "owner_count_median", "horsepower_median", "combined_fuel_economy",
                   "back_legroom_median", "front_legroom_median", "fuel_tank_volume_median",
                   "height_median", "length_median", "width_median", "pct_isCab", "pct_is_new"]
colonne_log = ["n_listings", "listings_per_day", "listing_span_days", "daysonmarket_median", "price_median"]

df_cluster = df_clean[colonne_cluster].copy()
df_cluster[colonne_log] = np.log1p(df_cluster[colonne_log])
df_cluster = pd.DataFrame(StandardScaler().fit_transform(df_cluster),
                          index=df_cluster.index, columns=df_cluster.columns)
df_cluster = df_cluster.rename(columns={c: "log1p_" + c for c in colonne_log})

assert df_clean.isna().sum().sum() == 0, "ci sono ancora NaN in df_clean"
assert df_cluster.isna().sum().sum() == 0, "ci sono ancora NaN in df_cluster"

df_clean.to_csv("../data/processed/seller_clean.csv")
df_cluster.to_csv("../data/processed/seller_clustering_base.csv")
print("seller_clean.csv:", df_clean.shape, "| seller_clustering_base.csv:", df_cluster.shape)

### Riepilogo delle decisioni (da mettere a fine sezione nel report)

| Variabile | Problema | Azione | Motivazione |
|---|---|---|---|
| 7 colonne numeriche | valori corrotti (separatore delle migliaia) | ricostruzione univoca, altrimenti NaN | 99,7% recuperato senza distorsioni |
| `sp_id` | letto come decimale, è un ID | intero, usato come indice | non è un predittore |
| `dominant_transmission` | codici poco leggibili (a, m) | rinominati | leggibilità |
| legroom, consumi, km | valori impossibili o incoerenti | → NaN, poi imputati | errori, non valori veri |
| auto d'epoca, elettriche, van, mega-venditori | outlier | tenuti | valori veri e interpretabili |
| `dominant_wheel_system` | 76% NaN | eliminata | imputazione non credibile |
| `pct_*`, `owner_count` dei venditori di solo nuovo | NaN in blocco | → 0 ("non applicabile") | un'auto nuova non ha proprietari né precedenti |
| `engine_displacement_median` | 24,5% NaN | regressione + colonna 0/1 | errore minimo sul 20% nascosto |
| NaN residui (< 1%) | sparsi | mediana per carrozzeria | le dimensioni dipendono dalla carrozzeria |
| alimentazione/cambio "unknown" | pochissimi casi | moda | impatto nullo |
| colore "unknown" | 11%, mancante mascherato | categoria a sé | non ricostruibile |
| conteggi, prezzi, durate | forte asimmetria | log1p (per il clustering) | asimmetria e outlier ridotti |
| carrozzeria, colore | categorie rare | raggruppate | troppo pochi casi |
| `pct_salvage`, `pct_theft_title` | quasi sempre 0 | binarizzate | conta la presenza |
| `year`, `n_listing_days`, city/highway | \|ρ\| ≥ 0,94 | eliminate o fuse | ridondanti |
| `savings_amount_median` | ρ ≈ 0,91 con `discount_ratio` | esclusa dal clustering | ridondante |
| `dominant_fuel_type` | ~99% gasoline | eliminata | quasi costante |
| `new_used_profile` | derivata da `pct_is_new` | esclusa dai modelli | ridondante |